In [ ]:
# ============================================================================
# DIAGNOSTIC CELL: ROLLING ADF WINDOW SENSITIVITY ANALYSIS
# ============================================================================
# Compares rolling ADF stationarity detection across window lengths
# 5, 10, 15, and 60 days. Does NOT modify any backtest logic.
# ============================================================================

#============================================================================
# THIS IS A TEST FILE WHICH INCLUDES THE FULL DIAGNOSTIC CODE FOR EVALUATION PURPOSES.
# WOULD NOT BE USED IN PRODUCTION.
# ALL DATA COMES FROM MAIN PCA.IPYNB SCRATCHPAD AND IS NOT RELOADED HERE.
#============================================================================

import time
import warnings

tenors_diag = ['1Mo', '3Mo', '6Mo', '1Yr', '2Yr', '3Yr', '5Yr', '7Yr', '10Yr', '30Yr']

regime_events = {
    'GFC':              ('2008-09-01', '2009-03-31'),
    'Euro Debt Crisis': ('2011-07-01', '2011-12-31'),
    'Taper Tantrum':    ('2013-05-01', '2013-09-30'),
    'COVID Shock':      ('2020-02-15', '2020-04-30'),
    'Fed Hiking Cycle': ('2022-01-01', '2022-12-31'),
}

# ── PART A: Compute rolling ADF for windows 5, 10, 15 days ──────────────────
print(f"\n{'='*80}")
print("PART A — Rolling ADF: windows 5, 10, 15 days")
print(f"{'='*80}")
print("  maxlag=0, autolag=None, regression='c'")
print("  (AIC lag selection is unreliable on windows <30 observations)")

adf_dfs = {60: rolling_adf_df}

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for w in [5, 10, 15]:
        t0 = time.time()
        pvals_w = {}
        for tenor in tenors_diag:
            series     = residuals_df[tenor].dropna()
            pvals      = []
            roll_dates = []
            for i in range(w, len(series) + 1):
                window_series = series.iloc[i - w:i]
                try:
                    result = adfuller(window_series, maxlag=0, autolag=None, regression='c')
                    pvals.append(result[1])
                except Exception:
                    pvals.append(np.nan)
                roll_dates.append(series.index[i - 1])
            pvals_w[tenor] = pd.Series(pvals, index=roll_dates)
        adf_dfs[w] = pd.DataFrame(pvals_w)
        print(f"  Window {w:>2}d: {len(adf_dfs[w])} dates  ({time.time() - t0:.1f}s)")

rolling_adf_5d  = adf_dfs[5]
rolling_adf_10d = adf_dfs[10]
rolling_adf_15d = adf_dfs[15]
print("  Stored as: rolling_adf_5d, rolling_adf_10d, rolling_adf_15d")


# ── Helper ────────────────────────────────────────────────────────────────────
def compute_episode_stats(pval_series, threshold=0.05):
    # Returns (pct_nonstat, avg_dur, max_dur, n_episodes) for p > threshold.
    valid = pval_series.dropna()
    if len(valid) == 0:
        return 0.0, 0.0, 0, 0
    nonstat  = (valid > threshold).astype(int).values
    pct      = nonstat.mean() * 100
    episodes = []
    ep_len   = 0
    for v in nonstat:
        if v:
            ep_len += 1
        else:
            if ep_len > 0:
                episodes.append(ep_len)
                ep_len = 0
    if ep_len > 0:
        episodes.append(ep_len)
    n_ep  = len(episodes)
    avg_d = float(np.mean(episodes)) if episodes else 0.0
    max_d = int(max(episodes))       if episodes else 0
    return pct, avg_d, max_d, n_ep


# ── PART B: Sensitivity analysis ─────────────────────────────────────────────
print(f"\n{'='*80}")
print("PART B — Sensitivity analysis: non-stationarity detection metrics (p > 0.05)")
print(f"{'='*80}")

hdr = f"  {'Tenor':<6}  {'Window':>7}  {'%NonStat':>9}  {'AvgDur(d)':>10}  {'MaxDur(d)':>10}  {'Episodes':>9}"
div = "  " + "-" * 62
print(hdr)

for tenor in tenors_diag:
    print(div)
    first = True
    for w in [5, 10, 15, 60]:
        df_w = adf_dfs[w]
        if tenor not in df_w.columns:
            continue
        pct, avg_d, max_d, n_ep = compute_episode_stats(df_w[tenor])
        label = tenor if first else ''
        print(f"  {label:<6}  {str(w)+'d':>7}  {pct:>8.1f}%  {avg_d:>10.1f}  {max_d:>10d}  {n_ep:>9d}")
        first = False
print(div)


# ── PART C: Regime event analysis ─────────────────────────────────────────────
print(f"\n{'='*80}")
print("PART C — Regime event analysis: 10Yr tenor detection timing")
print(f"{'='*80}")

for event_name, (ev_start, ev_end) in regime_events.items():
    ev_start_ts = pd.Timestamp(ev_start)
    ev_end_ts   = pd.Timestamp(ev_end)

    print(f"\n  ── {event_name}  ({ev_start} → {ev_end}) ──")
    print(f"  {'Window':>8}  {'First Detect':>13}  {'Lag (trd days)':>15}  {'Consec Days':>12}")
    print(f"  {'-'*56}")

    for w in [5, 10, 15, 60]:
        df_w = adf_dfs[w]
        if '10Yr' not in df_w.columns:
            continue
        mask      = (df_w.index >= ev_start_ts) & (df_w.index <= ev_end_ts)
        ev_series = df_w.loc[mask, '10Yr'].dropna()

        if len(ev_series) == 0:
            print(f"  {str(w)+'d':>8}  {'No data':>13}  {'—':>15}  {'—':>12}")
            continue

        nonstat_mask = ev_series > 0.05
        if not nonstat_mask.any():
            print(f"  {str(w)+'d':>8}  {'Not detected':>13}  {'—':>15}  {'—':>12}")
            continue

        first_detect  = ev_series[nonstat_mask].index[0]
        trd_day_lag   = int((ev_series.index < first_detect).sum())
        consec        = 0
        for v in nonstat_mask.loc[first_detect:].values:
            if v:
                consec += 1
            else:
                break
        print(f"  {str(w)+'d':>8}  {first_detect.strftime('%Y-%m-%d'):>13}  {trd_day_lag:>15d}  {consec:>12d}")

    # p-value table sampled every 5 trading days
    all_event_dates = sorted({
        dt for w in [5, 10, 15, 60]
        for dt in adf_dfs[w].index
        if ev_start_ts <= dt <= ev_end_ts
    })
    sample_dates = all_event_dates[::5] if len(all_event_dates) > 20 else all_event_dates

    print(f"\n  p-value table (every 5 trading days  |  * = non-stationary p > 0.05)")
    print(f"  {'Date':<12}" + "".join(f"  {str(w)+'d':>9}" for w in [5, 10, 15, 60]))
    for dt in sample_dates:
        row = f"  {dt.strftime('%Y-%m-%d'):<12}"
        for w in [5, 10, 15, 60]:
            df_w = adf_dfs[w]
            if dt in df_w.index and '10Yr' in df_w.columns:
                val    = df_w.loc[dt, '10Yr']
                marker = '*' if (not np.isnan(val) and val > 0.05) else ' '
                row   += f"  {val:>8.4f}{marker}"
            else:
                row   += f"  {'—':>9}"
        print(row)


# ── PART D: False trigger analysis ────────────────────────────────────────────
print(f"\n{'='*80}")
print("PART D — False triggers: non-stationary episodes < 3 consecutive days")
print("         outside the five known regime event windows")
print(f"{'='*80}")

regime_mask = pd.Series(False, index=residuals_df.index)
for ev_start, ev_end in regime_events.values():
    regime_mask.loc[ev_start:ev_end] = True

years_in_sample = (residuals_df.index[-1] - residuals_df.index[0]).days / 365.25

def count_false_triggers(pval_series, threshold=0.05, min_consec=3):
    # Count non-stationary episodes < min_consec days outside known regime events.
    valid   = pval_series.dropna()
    nonstat = (valid > threshold).astype(int)
    outside = nonstat[~regime_mask.reindex(nonstat.index, fill_value=False)]
    ft_count = 0
    ep_len   = 0
    for v in outside.values:
        if v:
            ep_len += 1
        else:
            if 0 < ep_len < min_consec:
                ft_count += 1
            ep_len = 0
    if 0 < ep_len < min_consec:
        ft_count += 1
    return ft_count

false_trigger_counts = {}

print(f"\n  Total false trigger counts")
print(f"  {'Tenor':<8}" + "".join(f"  {str(w)+'d':>8}" for w in [5, 10, 15, 60]))
print(f"  {'-'*50}")
for tenor in tenors_diag:
    row = f"  {tenor:<8}"
    for w in [5, 10, 15, 60]:
        df_w = adf_dfs[w]
        if tenor not in df_w.columns:
            row += f"  {'—':>8}"
            continue
        ft = count_false_triggers(df_w[tenor])
        false_trigger_counts[(tenor, w)] = ft
        row += f"  {ft:>8d}"
    print(row)

print(f"\n  False triggers per year  (sample span: {years_in_sample:.1f} years)")
print(f"  {'Tenor':<8}" + "".join(f"  {str(w)+'d':>8}" for w in [5, 10, 15, 60]))
print(f"  {'-'*50}")
for tenor in tenors_diag:
    row = f"  {tenor:<8}"
    for w in [5, 10, 15, 60]:
        ft = false_trigger_counts.get((tenor, w))
        row += f"  {ft / years_in_sample:>8.1f}" if ft is not None else f"  {'—':>8}"
    print(row)


# ── PART E: Visualization ─────────────────────────────────────────────────────
print(f"\n{'='*80}")
print("PART E — Plotly charts: 10Yr rolling ADF p-values across regime events")
print(f"{'='*80}")

window_colors = {5: '#e41a1c', 10: '#ff7f00', 15: '#4daf4a', 60: '#377eb8'}
window_names  = {5: '5d', 10: '10d', 15: '15d', 60: '60d (existing)'}

for event_name, (ev_start, ev_end) in regime_events.items():
    ev_start_ts = pd.Timestamp(ev_start)
    ev_end_ts   = pd.Timestamp(ev_end)

    fig = go.Figure()
    for w in [5, 10, 15, 60]:
        df_w = adf_dfs[w]
        if '10Yr' not in df_w.columns:
            continue
        mask   = (df_w.index >= ev_start_ts) & (df_w.index <= ev_end_ts)
        x_vals = df_w.index[mask]
        y_vals = df_w.loc[mask, '10Yr']
        fig.add_trace(go.Scatter(
            x=x_vals, y=y_vals,
            mode='lines',
            name=window_names[w],
            line=dict(color=window_colors[w], width=2.5 if w == 60 else 1.5),
            hovertemplate=f'<b>{window_names[w]}:</b> p=%{{y:.4f}}<extra></extra>'
        ))

    fig.add_hline(
        y=0.05, line_dash='dash', line_color='black', line_width=1.5,
        annotation_text='p = 0.05',
        annotation_position='top right',
        annotation_font=dict(size=11)
    )
    fig.update_layout(
        title=dict(
            text=(
                f'<b>Rolling ADF p-Values — 10Yr Tenor — {event_name}</b><br>'
                f'<span style="font-size:12px">'
                f'Above dashed line = non-stationary  |  '
                f'Shorter window = faster detection, more noise'
                f'</span>'
            ),
            x=0.5, font=dict(size=15)
        ),
        xaxis_title='Date',
        yaxis=dict(title='ADF p-value', range=[0, 1.05], tickformat='.2f'),
        template='plotly_white',
        hovermode='x unified',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        margin=dict(l=50, r=50, t=110, b=50),
        height=450
    )
    fig.show()
    print(f"  Rendered: {event_name}")


# ── PART F: Recommendation ────────────────────────────────────────────────────
print(f"\n{'='*80}")
print("PART F — Recommended window length per tenor")
print(f"{'='*80}")
print("  Rule: prefer shortest window with false triggers < 5/year")
print("        AND COVID shock detected within 10 trading days.")
print("        If no window meets both, flag for manual review.")
print()

covid_start_ts = pd.Timestamp('2020-02-15')
covid_end_ts   = pd.Timestamp('2020-04-30')

print(f"  {'Tenor':<8}  {'Recommended':>13}  Details")
print(f"  {'-'*75}")

for tenor in tenors_diag:
    recommendation = None
    chosen_reason  = ''
    all_details    = []

    for w in [5, 10, 15, 60]:
        df_w = adf_dfs[w]
        if tenor not in df_w.columns:
            continue
        ft   = false_trigger_counts.get((tenor, w), 9999)
        rate = ft / years_in_sample

        covid_mask    = (df_w.index >= covid_start_ts) & (df_w.index <= covid_end_ts)
        covid_series  = df_w.loc[covid_mask, tenor].dropna()
        nonstat_covid = covid_series > 0.05
        if nonstat_covid.any():
            covid_lag = int((covid_series.index < covid_series[nonstat_covid].index[0]).sum())
        else:
            covid_lag = 9999

        all_details.append(f"{w}d: FT={rate:.1f}/yr lag={covid_lag}d")

        if recommendation is None and rate < 5.0 and covid_lag <= 10:
            recommendation = w
            chosen_reason  = f"FT={rate:.1f}/yr, COVID lag={covid_lag} trd days"

    if recommendation is not None:
        print(f"  {tenor:<8}  {str(recommendation)+'d':>13}  {chosen_reason}")
    else:
        detail_str = '  |  '.join(all_details)
        print(f"  {tenor:<8}  {'MANUAL REVIEW':>13}  ⚠  {detail_str}")

print(f"\n{'='*80}")
print("DIAGNOSTIC COMPLETE — awaiting confirmation before modifying backtest logic")
print(f"{'='*80}")
